## PRUEBA TÉCNICA

Nombre: Darlyn Ludeña

# Etapa 1

Objetivo: Dado un usuario y su historial, rankear una lista de productos candidatos de
mayor a menor probabilidad de ser comprados en su próxima orden.

Contexto del problema

En una plataforma como e-commerce, predecir si un producto se compra o no es
insuficiente — necesitas ordenar miles de productos y mostrar los más relevantes primero.
Eso es un problema de ranking, no de clasificación binaria.

Primero cargamos los módulos con las diferentes funciones a emplear en la limpieza y creación de features.

In [1]:
from Etapa1.merge import *
from Etapa1.load import load_data
from Etapa1.no_comprados import *
from Etapa1.data_limpia_features import *
from Etapa1.model import *

In [2]:
order_products_prior,order_products_train, orders, products = load_data()

In [3]:
base=merge_tablas(order_products_prior, orders, products,"prior")

In [4]:
base

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32434484,3421083,39678,6,1,25247,prior,24,2,6,21.0,Free & Clear Natural Dishwasher Detergent,74,17
32434485,3421083,11352,7,0,25247,prior,24,2,6,21.0,Organic Mini Sandwich Crackers Peanut Butter,78,19
32434486,3421083,4600,8,0,25247,prior,24,2,6,21.0,All Natural French Toast Sticks,52,1
32434487,3421083,24852,9,1,25247,prior,24,2,6,21.0,Banana,24,4


In [5]:
base=labeling(base)
conteo_comprados_u_dep=conteo_comprados_user_dep(base)

In [6]:
dic_base,dic_products,dic_diferencia=calculo_diferentes(products,base)
base_no_comprados=base_final_no_comprados(dic_diferencia)
base_final=base_concatenada_comprados_no_comprados(base,base_no_comprados)
base_final=creacion_features(base_final)


A continuación entrenamos el modelo con la data de entrenamiento construida y emplearemos el ranker conocido como **XGBRanker con objective rank:ndcg**

Para luego validar el  modelo con `order_products__train.csv` como set de test (última orden real de cada
usuario) y calculamos la métrica NDCG@10.

In [7]:
X,y,base_final_model=variables_modelo(base_final)
ranker,group=modelo_grupo(base_final_model)
ranker=entrenar_modelo(ranker,X,y,group)

In [8]:
base_t=merge_tablas(order_products_train, orders, products,"train")
base_t=labeling(base_t)
conteo_comprados_u_dep_t=conteo_comprados_user_dep(base_t)
dic_base_t,dic_products_t,dic_diferencia_t=calculo_diferentes(products,base_t)
base_no_comprados_t=base_final_no_comprados(dic_diferencia_t)
base_final_t=base_concatenada_comprados_no_comprados(base_t,base_no_comprados_t)
base_final_t=creacion_features(base_final_t)

In [9]:
X_test,y_test,base_final_model_t=variables_modelo(base_final_t)
ranker_t,group_t=modelo_grupo(base_final_model_t)
scores = ranker.predict(X_test)
base_final_t["score"] = scores


In [10]:
ranking_t=create_ranking(base_final_t,score="score")
ndcg10_t=metricas_modelo(ranking_t,score="score")

Finalmente,rankeamos por popularidad global (product_global_popularity descendente) y comparar contra el modelo entrenado.

In [11]:
base_final_t["score_b"] = base_final_t["product_global_popularity"]
ranking_b=create_ranking(base_final_t,score="score_b")
ndcg10_b=metricas_modelo(ranking_b,score="score_b")


In [12]:
print( ndcg10_t,ndcg10_b)

0.12566075873478083 0.009743905585883032


Modelo |	NDCG@10 |
-------|-------|
Baseline (popularidad) |	0.0097|
XGBRanker |	0.12 |


El modelo XGBRanker presenta un aumento por sobre del 100% de la métrica obtenida en baseline, por lo que aprender preferencias individuales aporta valor frente a recomendar únicamente los productos más populares.


# Etapa 2

Objetivo: Dado un producto, encontrar los K productos más similares usando
representaciones vectoriales.



In [13]:
from Etapa2.sentence_transformer_similar_product import *

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Entrenaremos embeddings de productos basados en co-ocurrencia en órdenes (productos que se
compran juntos tienden a ser similares). 

In [14]:
model=SentenceTransformer("all-MiniLM-L6-v2")
embeddings=model.encode(products["product_name"].tolist(),show_progress_bar=True)

Batches: 100%|██████████| 1553/1553 [00:58<00:00, 26.55it/s]


In [15]:
products

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13
...,...,...,...,...
49683,49684,"Vodka, Triple Distilled, Twist of Vanilla",124,5
49684,49685,En Croute Roast Hazelnut Cranberry,42,1
49685,49686,Artisan Baguette,112,3
49686,49687,Smartblend Healthy Metabolism Dry Cat Food,41,8


In [16]:
products[products["product_id"]==1] 

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19


Función `get_similar_products(product_id, k=10)` que retorna los K más similares con su score de similitud coseno.

In [17]:
resultado = get_similar_products(1, top_k=5,embeddings=embeddings)

In [18]:
resultado

,product_id,product_name,aisle_id,department_id,similarity
31935,31936,Chocolate Chip Cookie Sandwich,61,19,0.873917
12480,12481,Oreo Chocolate Sandwich Cookies,61,19,0.873528
15758,15759,Peanut Butter Sandwich Cookies,61,19,0.869736
20948,20949,Vanilla Sandwich Cookies,37,1,0.867371
23931,23932,Chocolate Creme Sandwich Cookies,61,19,0.863877


Se puede observar que el producto es "Chocolate Sandwich Cookies" y los productos similares son otros tipos de galletas de chocolate, lo que tiene sentido dado el nombre del producto.

# Etapa 3

Objetivo: Construir un agente simple que, dado un mensaje en lenguaje natural, orqueste
las etapas anteriores y devuelva una recomendación al usuario.

Para ello, aplicaremos las funcionalidades del framework de langchain, y se implementa un agente conversacional que emplea un prompt con los pasos a seguir y las herramientas a emplear que son: `tool_get_user_history`, `tool_get_similar_products` y `tool_predict_reorder`.

Se recomienda que para el funcionamiento del agente conversacional se incluya la clave de la api de openai dentro del archivo `.env`, razón por la cual debido a su costo, solamente se incluye la sentencia: `key=CLAVE AQUI`.

In [20]:
from Etapa3.langchainscript import *

Batches: 100%|██████████| 1553/1553 [01:02<00:00, 24.81it/s]


In [21]:
tools = [
    tool_get_user_history, 
    tool_get_similar_products, 
    tool_predict_reorder
]

In [22]:
# Esto busca el archivo .env, lee tu OPENAI_API_KEY y la configura en el sistema automáticamente
load_dotenv()

# OPCIONAL: Línea de verificación para asegurarte de que se cargó correctamente antes de correr el agente
if not os.environ.get("key"):
    raise ValueError(" Error: No se encontró la variable OPENAI_API_KEY. Verifica tu archivo .env")
else:
    print("API Key de OpenAI cargada exitosamente desde el archivo .env")

ValueError:  Error: No se encontró la variable OPENAI_API_KEY. Verifica tu archivo .env

In [23]:
key=os.getenv("key")

In [24]:
products

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13
...,...,...,...,...
49683,49684,"Vodka, Triple Distilled, Twist of Vanilla",124,5
49684,49685,En Croute Roast Hazelnut Cranberry,42,1
49685,49686,Artisan Baguette,112,3
49686,49687,Smartblend Healthy Metabolism Dry Cat Food,41,8


In [25]:


# 1. Definimos el LLM (gpt-4o-mini es ideal por costo, velocidad y soporte de tools)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,api_key=key)



system_prompt = """Eres un Agente Conversacional experto en recomendación de productos de supermercado.
Tu objetivo es orquestar las herramientas disponibles para armar una recomendación personalizada.

FLUJO DE TRABAJO OBLIGATORIO:
1. Extrae o asume el `user_id` del contexto (asume el ID numérico 1 si el usuario no especifica uno).
2. Llama a `tool_get_user_history(user_id)`.
3. Para CADA nombre de producto que te devuelva el historial:
   a. Llama a `tool_get_product_id_by_name(product_name)` para obtener su `product_id` numérico.
   b. Si el ID es válido (diferente de -1), llama a `tool_get_similar_products(product_id)`.
4. Ordena los resultados finales de mayor a menor.

MANEJO DE CASOS EDGE (CRÍTICO PARA EVALUACIÓN):
- Si el usuario no tiene historial (lista vacía) o el producto no es encontrado en las herramientas (ID = -1), indícalo amigablemente diciendo que al no tener registros suficientes, le darás una cesta saludable genérica popular de la tienda (inventa 3 saludables como Avena, Manzana, Yogur Griego y asígnales scores estimados lógicos basados en popularidad global como 0.94, 0.91, 0.88).

Formatea tu respuesta final EXACTAMENTE con esta estructura de ejemplo:
"Basado en tu historial y productos similares, te recomiendo:

1. [Nombre del Producto] (score: 0.XX)
2. [Nombre del Producto] (score: 0.XX)
3. [Nombre del Producto] (score: 0.XX)"
"""




OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 

In [ ]:

# 4. Compilamos el Agente usando LangGraph con el parámetro 'prompt' corregido
agente_orquestador = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

In [ ]:


def probar_agente(mensaje_usuario: str):
    print(f"\n--- USUARIO: \"{mensaje_usuario}\" ---")
    
    # Se ejecuta invocando el grafo con el formato de historial de mensajes obligatorio
    resultado = agente_orquestador.invoke({
        "messages": [("user", mensaje_usuario)]
    })
    
    # Extraemos la última respuesta generada en la cadena del modelo
    respuesta_final = resultado["messages"][-1].content
    print(f"AGENTE:\n{respuesta_final}\n")

# Ejecución del Caso de Éxito Solicitado (Flujo feliz)
probar_agente("Quiero armar una cesta saludable para el desayuno")